In [ ]:
import deepxde as dde
import numpy as np
import pandas as pd
import torch

from matplotlib import pyplot as plt

dde.config.set_default_float("float32")
dde.config.set_random_seed(42)

rng = np.random.default_rng(42)

In [ ]:
log_nu = dde.Variable(np.log(0.1), dtype=torch.float32)

def viscosity():
    return torch.exp(log_nu)

H = 1.0
G = 1.0
T = 5.0
nu_true = 0.1

In [ ]:
geometric_interval = dde.geometry.Interval(-H, H)
time_interval = dde.geometry.TimeDomain(0.0, T)

full_domain = dde.geometry.GeometryXTime(geometric_interval, time_interval)

In [ ]:
def analytic_solution (x, nu=nu_true, max_iter=100):
    y = x[:, 0:1]
    t = x[:, 1:2]

    u_static = (G * (H ** 2) / (2 * nu)) * (1 - (y / H) ** 2)

    u_movement = np.zeros_like(y)

    for n in range(max_iter):
        k = (2 * n + 1) * np.pi / (2 * H)
        coeff = (-1)**n / (2 * n + 1)**3
        u_movement += coeff * np.cos(k * y) * np.exp(-nu * k**2 * t)
    
    u_movement *= (4 * G * H**2) / (nu * np.pi**3)
    
    return u_static - u_movement


In [ ]:
def pde(x, y):
    dy_t = dde.grad.jacobian(y, x, i=0, j=1)
    dy_yy = dde.grad.hessian(y, x, i=0, j=0)

    return dy_t - viscosity() * dy_yy - G

In [ ]:
boundary_condition = dde.icbc.DirichletBC(
    full_domain,
    lambda x: np.zeros((len(x), 1)),
    lambda _, on_b: on_b
)

initial_condition = dde.icbc.IC(
    full_domain,
    lambda x: np.zeros((len(x), 1)),
    lambda _, on_i: on_i
)

In [ ]:
N_obs = 150
sigma = 0.01

n_early = int(0.7 * N_obs)
t_obs = np.concatenate([
    rng.uniform(0.0, T/2, n_early),
    rng.uniform(T/2, T, N_obs - n_early)
]).reshape(-1, 1)

y_grade = np.linspace(-H, H, 20001)
u_steady_grade = (G * H**2 / (2 * nu_true)) * (1 - (y_grade / H)**2)

cdf = np.cumsum(np.abs(u_steady_grade))
cdf /= cdf[-1]

y_observed = np.interp(rng.uniform(0, 1, N_obs), cdf, y_grade).reshape(-1, 1)

X_obs = np.hstack([y_observed, t_obs])
u_obs = analytic_solution(X_obs) + sigma * rng.standard_normal((N_obs, 1))

observe = dde.icbc.PointSetBC(X_obs, u_obs, component=0)

In [ ]:
data = dde.data.TimePDE(
    full_domain,
    pde,
    [boundary_condition, initial_condition, observe],
    num_domain=4000,
    num_boundary=200,
    num_initial=400,
    num_test=2000,
    solution=lambda x: analytic_solution(x, nu_true)
)

In [ ]:
net = dde.nn.FNN([2] + [128] * 4 + [1], "tanh", "Glorot uniform")
model = dde.Model(data, net)

weights = [10.0, 10.0, 10.0, 1.0]


In [ ]:
model.compile(
    "adam",
    lr=1e-3,
    loss_weights=weights,
    metrics=["l2 relative error"],
    external_trainable_variables=[log_nu]
)

var_callback = dde.callbacks.VariableValue(
    [log_nu],
    period=250,
    filename="poiseuille_variables.dat",
    precision=8
)

losshistory, train_state = model.train(
    iterations=15000,
    display_every=250,
    callbacks=[var_callback]
)

dde.optimizers.config.set_LBFGS_options(maxiter=30000, ftol=1e-14, gtol=1e-12)

model.compile(
    "L-BFGS",
    loss_weights=weights,
    external_trainable_variables=[log_nu]
)

losshistory_lbfgs, train_state_lbfgs = model.train(callbacks=[var_callback])

nu_pred = float(torch.exp(log_nu).detach().cpu().numpy())
rel_err = abs(nu_pred - nu_true) / nu_true

print(f"\n{'='*60}")
print(f"True:  nu_true = {nu_true:.6f}")
print(f"Prediction:     nu_pred = {nu_pred:.6f}")
print(f"Error: {rel_err:.2%}")
print(f"{'='*60}")

In [ ]:
losshistory, train_state = model.train(
    iterations=10000, display_every=250, callbacks=[var_callback]
)

lab_idx = {'PDE':0, 'BC':1, 'IC':2, 'DATA':3}
steps        = np.array(losshistory.steps)                
train_loss   = np.array(losshistory.loss_train)          
test_loss    = np.array(losshistory.loss_test)           
labels       = ['PDE', 'BC', 'IC', 'DATA']
weights      = np.array([10.0, 10.0, 10.0, 1.0])        

fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogy(steps, train_loss.sum(axis=1), 'b-',  label='Train total')
ax.semilogy(steps, test_loss.sum(axis=1),  'r--', label='Test total')
ax.set_xlabel('Step'); ax.set_ylabel('Weighted loss (log)')
ax.set_title('Total functional'); ax.legend(); ax.grid(True, which='both', ls=':')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
for ax, lab, w in zip(axes.flat, labels, weights):
    ax.semilogy(steps, train_loss[:, lab_idx[lab]] / w, 'b-',  label='Train')
    ax.semilogy(steps, test_loss[:,  lab_idx[lab]] / w, 'r--', label='Test')
    ax.set_ylabel('Unweighted loss'); ax.set_title(lab)
    ax.legend(); ax.grid(True, which='both', ls=':')
plt.tight_layout()
plt.show()